# Import Libraries

In [ ]:
import os
import csv
import json
import gzip
import pickle
import itertools
import numpy as np
import pandas as pd
import gseapy as gp
import seaborn as sns
from scipy import stats
import matplotlib.cm as cmx
import matplotlib.pyplot as plt
import matplotlib.lines as mlines
from datasets import load_dataset
from itertools import combinations
import matplotlib.colors as colors
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from pxblat import Server, Client #pxblat==0.3.6
from warnings import simplefilter

# Matrix Generation

## Load Data

In [ ]:
# Define model and paths
model = 'pretrained' # {pretrained, finetuned}
dataset = 'scgpt_ms' # {ms, pancreas}
base = "/Users/vivianschu/Documents/MBP/Lab/Genomic Interpretability/scgpt_attn_results/test"
approach = 'attention'

In [ ]:
data_dir = os.path.join(base, dataset, model)

pkl_files = [
    'examples_scores_attention_layer0.p',
    'examples_scores_attention_layer1.p',
    'examples_scores_attention_layer2.p',
    'examples_scores_attention_layer3.p',
    'examples_scores_attention_layer4.p',
    'examples_scores_attention_layer5.p',
    'examples_scores_attention_layer6.p',
    'examples_scores_attention_layer7.p',
    'examples_scores_attention_layer8.p',
    'examples_scores_attention_layer9.p',
    'examples_scores_attention_layer10.p',
    'examples_scores_attention_layer11.p',
]


In [ ]:
# Load all pickle files into memory
layers_data = []
for file_name in pkl_files:
    file_path = os.path.join(data_dir, file_name)
    with open(file_path, 'rb') as f:
        layers_data.append(pickle.load(f))

# Assuming all layers have the same number of heads and all heads have the same number of cells
num_layers = len(layers_data)
num_heads = len(layers_data[0])
num_cells = len(layers_data[0][0])
num_genes = len(layers_data[0][0][0][0])  # Assuming each cell contains data for the same number of genes
num_expression = len(layers_data[0][0][0][3])

print('Check Data:', num_layers, num_heads, num_cells, num_genes, num_expression)

## Mean Scores

In [ ]:
# Initialize an empty DataFrame to store the mean scores
mean_score_df: pd.DataFrame = pd.DataFrame()

# Iterate through each layer
for layer in range(num_layers):
    print("LAYER:", layer)
    
    # Load the pickled results for the current layer
    with open(f'{data_dir}/examples_scores_attention_layer{layer}.p', 'rb') as f:
        results: dict = pickle.load(f)
    # Dictionary to store mean scores for each head in the current layer
    tmp_dict: dict = {}
    # Iterate through each head in the results
    for head in results:
        print("HEAD:", head)
        tmp_list: list = []
        # Calculate mean score for each sequence in the current head
        for i in range(len(results[head])):
            tmp_list.append(np.mean(results[head][i][0]))
        # Store the mean scores for the current head in the dictionary
        tmp_dict[head] = tmp_list
    # Convert the dictionary to a DataFrame
    tmp_df: pd.DataFrame = pd.DataFrame(tmp_dict)
    # Rename the columns to indicate the head number
    tmp_df.columns = [f'head{i}' for i in range(num_heads)]
    # Add a column to indicate the layer
    tmp_df['layer'] = f'layer{layer}'
    # Concatenate the current layer's DataFrame with the main DataFrame
    mean_score_df = pd.concat([mean_score_df, tmp_df])

# Save the final DataFrame as a CSV file
mean_score_df.to_csv(f'{data_dir}/examples_mean_{approach}_scores.csv', index=False)

# Clean up to free memory
del(tmp_df, tmp_dict, tmp_list, results, head, layer, f, i)

### Heatmap

In [ ]:
# Calculate the mean score for each layer
tmp_mean_df = mean_score_df.groupby('layer').mean()
# Normalize the scores within each layer to the range [0, 1]
tmp_mean_df = tmp_mean_df.apply(lambda x: (x - x.min()) / (x.max() - x.min()), axis=1)
# Sort the DataFrame by layer numbers (assumes 'layer' column is formatted as 'layerX')
tmp_mean_df = tmp_mean_df.reindex(sorted(tmp_mean_df.index, key=lambda x: int(x[5:])))

# Set up the plot
plt.figure(1, figsize=(9, 6))
sns.set(color_codes=True)
sns.set(font_scale=0.9)

# Heatmap to visualize the normalized mean scores
ax = sns.heatmap(tmp_mean_df, cmap='GnBu', cbar_kws={'label': 'Scale'})
ax.set(ylabel="Layers", xlabel="Heads")
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, horizontalalignment='right')
ax.set_yticklabels(ax.get_yticklabels(), rotation=45)
# Save the heatmap as a PNG file
plt.savefig(f"{data_dir}/scgpt_mean_attention_heatmap.png", dpi=300, bbox_inches='tight')
# Display the heatmap
plt.show()

# Save the normalized mean scores to a CSV file
tmp_mean_df.to_csv(f'{data_dir}/tmp_mean_{approach}_scores.csv', index=False)
# Clean up
del(tmp_mean_df, ax)

## Combine Matrix

In [ ]:
# Function to concatenate items in a list into a comma-separated string
def concatenate_items(items):
    return ','.join(str(item) for item in items)

# Define the output CSV file path
output_csv_path = os.path.join(data_dir, 'compiled_cells.csv')

# Open the CSV file for writing
with open(output_csv_path, 'w', newline='') as csvfile:
    csvwriter = csv.writer(csvfile)
    
    # Write the header row
    headers = ['gene_sequence', 'label', 'expression'] + [f'layer{layer}_head{head}' for layer in range(num_layers) for head in range(num_heads)]
    csvwriter.writerow(headers)

    # Iterate over each cell
    for cell_index in range(num_cells):
        row = []
        
        # Initialize gene sequence and label
        gene_seq = ''
        label = ''
        expression_seq = ''

        for layer in range(num_layers):
            for head in range(num_heads):
                # Extract the relevant data for the current cell, layer, and head
                attn_scores = layers_data[layer][head][cell_index][0]
                gene_names = layers_data[layer][head][cell_index][1]
                label = layers_data[layer][head][cell_index][2]
                expression_vals = layers_data[layer][head][cell_index][3]

                # Concatenate the attention scores, gene names, and expression values into strings
                attn_seq = concatenate_items(attn_scores)
                gene_seq = concatenate_items(gene_names)
                expression_seq = concatenate_items(expression_vals)
                
                # Append the concatenated attention scores to the row
                row.append(attn_seq)

        # Write the row to the CSV file
        csvwriter.writerow([gene_seq, label, expression_seq] + row)

#### Check

In [ ]:
cells_path = os.path.join(data_dir, 'compiled_cells.csv')
df = pd.read_csv(cells_path)

df.head()

In [ ]:
# Get the first element (assuming comma-separated values)
gene_seq = df['gene_sequence'].iloc[0]

# # Split the string by comma, remove whitespace, and convert to float (handling errors)
num_genes = sum(1 for _ in [(x.strip()) for x in gene_seq.split(',')])
print(f'Number of Genes: {num_genes}')

exp_seq = df['expression'].iloc[0]
num_exp = sum(1 for _ in [(x.strip()) for x in exp_seq.split(',')])
print(f'Number of Expression Values: {num_exp}')

## Generate features

### Position

In [ ]:
cells_path = os.path.join(data_dir, 'compiled_cells.csv')
cells = pd.read_csv(cells_path)

cells.head()

In [ ]:
seq = df['gene_sequence'].iloc[0]
len_seq = sum(1 for _ in [(x.strip()) for x in seq.split(',')])

total_cells = len_seq

# Calculate the lengths of the thirds
third_length = total_cells // 3
last_third_start = 2 * third_length

# If total_cells is not perfectly divisible by 3, adjust the last third to include any extra cells
extra_cells = total_cells % 3
if extra_cells != 0:
    last_third_start += extra_cells - 1

# Generate sequences for first, middle, and last thirds
first_third = ["1"] * third_length + ["0"] * (total_cells - third_length)
middle_third = ["0"] * third_length + ["1"] * third_length + ["0"] * (total_cells - 2 * third_length)
last_third = ["0"] * last_third_start + ["1"] * (total_cells - last_third_start)

first_third_seq = ",".join(map(str, first_third))
middle_third_seq = ",".join(map(str, middle_third))
last_third_seq = ",".join(map(str, last_third))

cells['position_first_third'] = first_third_seq
cells['position_middle_third'] = middle_third_seq
cells['position_last_third'] = last_third_seq

In [ ]:
# Calculate total number of rows in the dataframe
total_rows = len(cells)

# # Generate the sequence of numbers from 1 to 499
sequence = ",".join(map(str, range(1, 501)))

# Create a list where each element is the sequence, replicated for each row in the dataframe
position_sequence = [sequence for _ in range(total_rows)]

# # Assign this list of lists to the 'position' column in your dataframe
cells['position'] = position_sequence

cells.head()

In [ ]:
# Export compiled_cells.csv
cells.to_csv(cells_path, index=False, sep=';')

### MSigDB Gene Sets

In [ ]:
# Import compiled_cells.csv
cells_path = os.path.join(data_dir, 'compiled_cells.csv')
cells = pd.read_csv(cells_path, sep=';')

cells.head()

In [ ]:
# Geneset files
genesets = ['H_geneset.csv', 'c8_geneset.csv', 'c5_geneset.csv']

In [ ]:
def generate_binary_string(gene_sequence, gene_set):
    # Convert gene_sequence into a list of genes
    ordered_genes = gene_sequence.split(',')
    # Generate binary string list where '1' indicates presence and '0' indicates absence in gene_set
    binary_string = ['1' if gene in gene_set else '0' for gene in ordered_genes]
    # Convert the binary string list to a comma-separated string
    binary_string = ','.join(binary_string)
    
    return binary_string

In [ ]:
simplefilter(action="ignore", category=pd.errors.PerformanceWarning)

# Iterate over genesets
for geneset in genesets:
    geneset_path = os.path.join(base, 'genesets', geneset)
    print('Geneset File Path:', geneset_path)

    # Read geneset file with specifying data types to avoid DtypeWarning
    geneset_raw = pd.read_csv(geneset_path, dtype={'column_name': str})
    # Extract unique gene set names
    gene_set_names = geneset_raw['gs_name'].unique()
    # Create a dictionary with gene set names as keys and their respective gene symbols as values
    gene_set_dict = {gs_name: set(geneset_raw.loc[geneset_raw['gs_name'] == gs_name, 'gene_symbol']) for gs_name in gene_set_names}
    # Apply the function for each gene set and append the results as new columns to cells DataFrame
    for gs_name, gene_set in gene_set_dict.items():
        cells[gs_name] = cells['gene_sequence'].apply(lambda x: generate_binary_string(x, gene_set))

    # Drop gene ddset file data from memory to conserve memory
    del geneset_raw

In [ ]:
# Export compiled_cells_features.csv
cells_features_path = os.path.join(data_dir, 'compiled_cells-features.csv')
cells.to_csv(cells_features_path, index=False, sep=';')

#### Filter columns
MS; at least 35 genes in 10% of samples

Pancreas; at least 40 genes in 10% of samples

In [ ]:
# Open compiled_cells.csv
compiled_cells_path = os.path.join(data_dir, 'compiled_cells.csv')
compiled_cells = pd.read_csv(compiled_cells_path, sep=';')

compiled_cells.head()

In [ ]:
# Open compiled_cells_features.csv
cells_features_path = os.path.join(data_dir, 'compiled_cells-features.csv')
cells_features = pd.read_csv(cells_features_path, sep=';')

cells_features.head()

In [ ]:
# Get the first element (assuming comma-separated values)
gene_str = cells_features['gene_sequence'].iloc[0]

# Split the string by comma, remove whitespace, and convert to float
num_floats = sum(1 for _ in [(x.strip()) for x in gene_str.split(',')])

print(f"Number of float values in 'expression[0]': {num_floats}")

columns_to_consider = ['gene_sequence'] + list(cells_features.columns[103:])
print('Number of Columns:', len(columns_to_consider), columns_to_consider)

# Filtering the dataframe to only include these columns
filtered_cells = cells_features[columns_to_consider]
print(filtered_cells.shape)

In [ ]:
# Create a list to store columns to drop
columns_to_drop = []

# Iterate over columns
for col in filtered_cells.columns:
    if col != 'gene_sequence': 
        #### REPLACE CONDITION ####
        count = filtered_cells[col].apply(lambda x: sum(int(i) for i in x.split(',')) < 35).sum() 
        #### REPLACE CONDITION ####
        
        if count / len(filtered_cells) > 0.10:
            columns_to_drop.append(col)  # Add column to drop list

# Drop columns outside the loop
filtered_cells.drop(columns_to_drop, axis=1, inplace=True)

In [ ]:
print('Shape:', filtered_cells.shape)
filtered_cells.head()

In [ ]:
# Export compiled_cells_all-features.csv
num_col = filtered_cells.shape[1]
filtered_cells_path = os.path.join(data_dir, f'filt_features_{num_col}.csv')
filtered_cells.to_csv(filtered_cells_path, index=False, sep=';')

## Export final filtered matrix with features

In [ ]:
# Merge the DataFrames on the 'gene_sequence' column
print('Number of Features:', num_col)
merge_filtered_cells = pd.merge(compiled_cells, filtered_cells, on='gene_sequence', how='inner')
merge_filtered_cells.head()

In [ ]:
final_df_path = os.path.join(data_dir, f'scgpt_{model}_{dataset}-features_{num_col}.csv')
merge_filtered_cells.to_csv(final_df_path, index=False, sep=';')

# Validation

In [ ]:
# Load attention scores and features
master_df = pd.read_csv(final_df_path, sep=';')
master_df.head()

In [ ]:
# Load the JSON data
with open(f'{base}/scgpt_{model}_{dataset}_coefficients.json', 'r') as file:
    coeff = json.load(file)

## Number of Highly Explainable Heads

In [ ]:
num_heads = 96
print(f'Total Number of Heads: {num_heads}')

In [ ]:
def count_biological_features(data, threshold=4):
    feature_counts = []
    
    for layer_head, content in data.items():
        count = 0
        sentences = content.get('sentences', [])
        
        for sentence in sentences:
            for item in sentence:
                if isinstance(item, list) and len(item) == 2:
                    feature, coefficient = item
                    if coefficient < -threshold or coefficient > threshold:
                        count += 1
        
        feature_counts.append(count)
    
    return feature_counts

# Calculate the number of biological features for each head
feature_counts = count_biological_features(coeff)

# Calculate the average number of biological features
average_features = np.mean(feature_counts)

# Find the maximum and second maximum number of biological features
max_features = np.max(feature_counts)
second_max_features = np.partition(feature_counts, -2)[-2]

print("Average number of biological features:", average_features)
print("Maximum number of biological features:", max_features)
print("Second maximum number of biological features:", second_max_features)

## Top & Bottom 3 Coefficients

In [ ]:
#### REPLACE CONDITION ####
layer = 6
head = 1
#### REPLACE CONDITION ####

In [ ]:
# Function to get top 3 positive and negative coefficients
def get_top_coeff(sentences):
    # Filter out sentences that are not lists or do not contain a numerical coefficient
    valid_sentences = [s for s in sentences if isinstance(s, list) and len(s) == 2 and isinstance(s[1], (int, float))]
    
    # Sort sentences by the coefficient
    sorted_sentences = sorted(valid_sentences, key=lambda x: x[1])
    
    # Get top 3 negative and top 3 positive coefficients
    top_negative = sorted_sentences[:3]
    top_positive = sorted_sentences[-3:]
    
    return top_positive, top_negative

# Process each layer and head pair
top_coeff = {}
for layer_head, content in coeff.items():
    sentences = content['sentences'][0]
    top_positive, top_negative = get_top_coeff(sentences)
    top_coeff[layer_head] = {
        'top_positive': top_positive,
        'top_negative': top_negative
    }

# Print the results
for layer_head, coefficients in top_coeff.items():
    print(f"{layer_head}:\nTop Positive: {coefficients['top_positive']}\nTop Negative: {coefficients['top_negative']}\n")

In [ ]:
# Top Coefficients
top_coeff[f'layer{layer}_head{head}']

## Sample-wise Correlation

In [ ]:
top = top_coeff[f'layer{layer}_head{head}']['top_positive']
features = [item[0] for item in top]
print(f'Features: {features}')

In [ ]:
def configure_scgpt(df):
    # Convert comma-separated string columns to numpy arrays of floats
    for col in df.columns:
        if col not in ['gene_sequence', 'label']:
            df[col] = df[col].apply(lambda x: np.array(x.split(','), dtype=float))
    
    return {
        'attention_score_columns': [col for col in df.columns if 'layer' in col and 'head' in col],
        'bio_feature_columns': [col for col in df.columns if not (('layer' in col and 'head' in col) or ('gene_sequence' in col) or ('label' in col))],
        'seq_length': 500  # This length should be adjusted based on your data.
    }

In [ ]:
df = master_df.copy()
config = configure_scgpt(df)

In [ ]:
def run_spearman(df, attention_score_column, bio_feature_columns):
    results = []
    # add position
    for index, seq in df.iterrows():
        x = np.concatenate(seq[bio_feature_columns].values).ravel().reshape((len(bio_feature_columns), 500)).T
        y = seq[attention_score_column]
        x = np.nan_to_num(x)

        # Spearman correlation for each feature column
        for feature in bio_feature_columns:
            result = stats.spearmanr(x[:, bio_feature_columns.index(feature)], y)
            if not np.isnan(result.statistic):
                # each position's data
                for pos in range(500):  # Assuming sequence length is 500
                    results.append({
                        'gene_sequence': seq['gene_sequence'],
                        'feature': feature,
                        'statistic': result.statistic,
                        'position': pos,
                        'attention_scores': y[pos]  
                    })

    return pd.DataFrame(results)

In [ ]:
attention_score_column = f'layer{layer}_head{head}'
results_df = run_spearman(df, attention_score_column, features)

In [ ]:
results_df['ymin'] = results_df.groupby('gene_sequence')['attention_scores'].transform('mean')
results_df_update = results_df
results_df_update['position2'] = results_df_update['position'] + 1  # assuming each position is a discrete point

In [ ]:
results_df_update.to_csv(f'{base}/scgpt_{model}_{dataset}_{attention_score_column}.csv')

In [ ]:
results_df.head()

## Plot Head-Feature Correlation

### Layer 6 Head 1

"layer6_head1": "Head Name: Membrane Transport Focus\n\nThis head is primarily focused on genes related to membrane protein complexes and transporter activity, indicating an attention towards processes involved in cellular transport mechanisms.",

In [ ]:
# Load attention score correlations
corr_path = f'{base}/scgpt_{model}_{dataset}_layer{layer}_head{head}.csv'
corr_df = pd.read_csv(corr_path)

print(f'Correlations Shape: {corr_df.shape}')
corr_df.head()

In [ ]:
# Sort the DataFrame by the 'statistic' column in descending order
sorted_corr = corr_df.sort_values(by='statistic', ascending=False)

# Drop duplicates to get unique 'statistic' values, keeping the first occurrence
unique_sorted_master = sorted_corr.drop_duplicates(subset=['feature'])

# Retrieve the top 3 rows with the highest unique values in the 'statistic' column
top_3_rows = unique_sorted_master.head(3)

top_3_rows = top_3_rows.reset_index(drop=True)

# Print the top 3 unique rows
print(top_3_rows)

In [ ]:
gene_seq = top_3_rows['gene_sequence']
print(gene_seq)
stat = top_3_rows['statistic']
print(stat)
feature = list(set(top_3_rows['feature']))
print(feature)

In [ ]:
cell_list = []

for idx, seq in enumerate(gene_seq):
    cell_df = master_df[master_df['gene_sequence'] == seq]
    index = cell_df.index[0]
    cell_list.append(index)

print(f'Cell List: {cell_list}')
num_cells = len(cell_list)

In [ ]:
# Number of cells to plot
num_cells = len(cell_list)

# Create subplots
fig, axes = plt.subplots(num_cells, 1, figsize=(12, 3 * num_cells))
fig.suptitle(f'Layer {layer} Head {head}', fontsize=13)

# Custom legend
blue_circle = mlines.Line2D([], [], color='blue', marker='o', linestyle='None', markersize=10, label=feature[0], alpha=0.5)
green_circle = mlines.Line2D([], [], color='green', marker='o', linestyle='None', markersize=10, label=feature[1], alpha=0.5)
red_circle = mlines.Line2D([], [], color='red', marker='o', linestyle='None', markersize=10, label='Overlap', alpha=0.5)
grey_circle = mlines.Line2D([], [], color='gainsboro', marker='o', linestyle='None', markersize=10, label='No Feature', alpha=0.5)
dark_line = mlines.Line2D([], [], color='slategrey', linestyle='--', label='Mean Attention')

fig.legend(handles=[dark_line, grey_circle, red_circle, blue_circle, green_circle], loc='upper center', bbox_to_anchor=(0.5, 0.95), ncol=5, frameon=False)


for i, cell_idx in enumerate(cell_list):
    print(f'Index: {i}')
    print(f'Cell Number: {cell_idx}')
    
    cell_corr = corr_df[corr_df['gene_sequence'] == gene_seq[i]][:500]
    cell_df = master_df[master_df['gene_sequence'] == gene_seq[i]]
    
    cell_corr = cell_corr[:500]
    
    feature_cell_df = cell_df[cell_df['gene_sequence'] == gene_seq[i]]

    feature_presence_1 = feature_cell_df[feature[0]][cell_idx].split(',')
    feature_presence_2 = feature_cell_df[feature[1]][cell_idx].split(',')

    cell_corr['feature_presence_1'] = feature_presence_1
    cell_corr['feature_presence_1'] = pd.to_numeric(cell_corr['feature_presence_1'], errors='coerce')
    cell_corr['feature_presence_2'] = feature_presence_2
    cell_corr['feature_presence_2'] = pd.to_numeric(cell_corr['feature_presence_2'], errors='coerce')
    attn_scores = cell_corr['attention_scores']
    posn = cell_corr['position2']

    presence_1 = cell_corr['feature_presence_1']
    presence_2 = cell_corr['feature_presence_2']

    mean_attn = attn_scores.mean()
    print(f'Sample Mean Attn: {mean_attn}')
    
    # Color Scheme
    # Set colors based on feature and overlap
    colors = []

    for p1, p2 in zip(presence_1, presence_2):
        if (p1 == 1 and p2 == 1):
            colors.append('red')
        elif p1 == 1:
            colors.append('blue')
        elif p2 == 1:
            colors.append('green')
        else:
            colors.append('gainsboro')

    ax = axes[i] if num_cells > 1 else axes
    ax.scatter(posn, attn_scores, c=colors, alpha=0.5)
    ax.axhline(y=mean_attn, color='slategrey', linewidth=1, linestyle='--', label='Mean Attention')
    # ax.set_ylabel('Attention Score')
    ax.set_title(f'Cell {cell_idx}')
    ax.tick_params(axis='x', which='both', bottom=False, top=False, labelbottom=False)

fig.text(-0.02, 0.5, 'Attention Score', va='center', rotation='vertical')
plt.tight_layout(rect=[0, 0, 1, 0.95])
fig.savefig(f'layer{layer}_head{head}.png', format='png', dpi=300)
plt.show()